In [ ]:
import os
import re
import csv

# ----------------------------- CONFIG -----------------------------
ROOT = r"F:\ISL_CSLRT_Corpus\Sentence"   # folder that contains all sentence folders
DRY_RUN = False                           # True = preview only, False = really rename
LOG_FILE = "rename_log.csv"
# ------------------------------------------------------------------

def sort_key(name):
    """free, free_(2), free_(3) ... stay together and in order"""
    m = re.match(r"^(.*?)(?:_?\((\d+)\))?$", name)
    return (m.group(1).lower(), int(m.group(2)) if m.group(2) else 1)

log_rows = []
sentences = sorted(d for d in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, d)))

for s in sentences:
    s_path = os.path.join(ROOT, s)
    subs = sorted((d for d in os.listdir(s_path) if os.path.isdir(os.path.join(s_path, d))), key=sort_key)

    # skip if already numbered 1..N
    if subs == [str(i) for i in range(1, len(subs) + 1)]:
        print(f"(already numbered) {s}")
        continue

    plan = [(old, str(i)) for i, old in enumerate(subs, start=1)]
    for old, new in plan:
        print(f"{s}: {old}  ->  {new}")
        log_rows.append([s, old, new])

    if not DRY_RUN:
        # two-step rename so it never collides with an existing folder "1", "2", ...
        for old, _ in plan:
            os.rename(os.path.join(s_path, old), os.path.join(s_path, f"__tmp__{old}"))
        for old, new in plan:
            os.rename(os.path.join(s_path, f"__tmp__{old}"), os.path.join(s_path, new))

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["sentence", "old_folder_name", "new_folder_name"])
    w.writerows(log_rows)

print(f"\n{len(sentences)} sentence folders processed. Log saved to {LOG_FILE}")
if DRY_RUN:
    print("DRY RUN: nothing was renamed. Set DRY_RUN = False and run the cell again to apply.")


(already numbered) He_is_going_into_the_room
(already numbered) No_need_to_worry_dont_worry
(already numbered) This_place_is_beautiful
(already numbered) are_you_free_today
(already numbered) are_you_hiding_something
(already numbered) bring_water_for_me
(already numbered) can_i_help_you
(already numbered) can_you_repeat_that_please
(already numbered) comb_your_hair
(already numbered) congratulations
(already numbered) could_you_please_talk_slower
(already numbered) do_me_a_favour
(already numbered) do_not_abuse_him
(already numbered) do_not_be_stubborn
(already numbered) do_not_hurt_me
(already numbered) do_not_make_me_angry
(already numbered) do_not_take_it_to_the_heart
(already numbered) do_not_worry
(already numbered) do_you_need_something
(already numbered) go_and_sleep
(already numbered) had_your_food
(already numbered) he_came_by_train
(already numbered) he_is_on_the_way
(already numbered) he_she_is_my_friend
(already numbered) he_would_be_coming_today
(already numbered) help_me

In [11]:
import os, re, csv

ROOT = r"F:\ISL_CSLRT_Corpus\Word"    # folder that contains all word folders
DRY_RUN = False                        # True = preview only, False = really rename
LOG_FILE = "word_rename_log.csv"
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

nat = lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]
rows = []

for word in sorted(d for d in os.listdir(ROOT) if os.path.isdir(os.path.join(ROOT, d))):
    folder = os.path.join(ROOT, word)
    imgs = sorted((f for f in os.listdir(folder) if f.lower().endswith(IMG_EXT)), key=nat)
    plan = [(old, f"{i}{os.path.splitext(old)[1].lower()}") for i, old in enumerate(imgs, 1)]
    rows += [[word, old, new] for old, new in plan]
    print(f"{word}: {len(plan)} images")

    if not DRY_RUN:   # two-step rename to avoid name clashes
        for old, _ in plan: os.rename(os.path.join(folder, old), os.path.join(folder, "__tmp__" + old))
        for old, new in plan: os.rename(os.path.join(folder, "__tmp__" + old), os.path.join(folder, new))

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["word", "old_name", "new_name"])
    w.writerows(rows)

print(f"\nLog saved to {LOG_FILE}" + ("  (DRY RUN: nothing renamed)" if DRY_RUN else ""))


A LOT: 4 images
ABUSE: 5 images
AFRAID: 6 images
AGREE: 7 images
ALL: 4 images
ANGRY: 10 images
ANYTHING: 4 images
APPRECIATE: 5 images
BAD: 5 images
BEAUTIFUL: 5 images
BECOME: 5 images
BED: 7 images
BORED: 6 images
BRING: 7 images
CHAT: 6 images
CLASS: 5 images
COLD: 7 images
COLLEGE_SCHOOL: 18 images
COMB: 7 images
COME: 6 images
CONGRATULATIONS: 7 images
CRYING: 13 images
DARE: 6 images
DIFFERENCE: 5 images
DILEMMA: 6 images
DISAPPOINTED: 6 images
DO: 12 images
DON'T CARE: 3 images
ENJOY: 4 images
FAVOUR: 4 images
FEVER: 7 images
FINE: 6 images
FOOD: 23 images
FREE: 7 images
FRIEND: 6 images
FROM: 7 images
GO: 9 images
GOOD: 7 images
GRATEFUL: 6 images
HAD: 4 images
HAPPENED: 5 images
HAPPY: 5 images
HEAR: 7 images
HEART: 6 images
HELLO_HI: 4 images
HELP: 26 images
HIDING: 6 images
HOW: 20 images
HUNGRY: 7 images
HURT: 14 images
I_ME_MINE_MY: 97 images
KIND: 7 images
LEAVE: 5 images
LIKE: 6 images
LIKE_LOVE: 7 images
MEAN IT: 3 images
MEDICINE: 6 images
MEET: 5 images
NAME: 7 image

In [ ]:
import os
import pandas as pd

DATASET_PATH = r"F:\ISL_CSLRT_Corpus"

rows = []

for task in ["Word", "Sentence"]:

    task_path = os.path.join(DATASET_PATH, task)

    if not os.path.isdir(task_path):
        print("Not found:", task_path)
        continue

    for label in os.listdir(task_path):

        label_path = os.path.join(task_path, label)

        if not os.path.isdir(label_path):
            continue

        # Check everything inside the label folder
        for item in os.listdir(label_path):

            item_path = os.path.join(label_path, item)

            # If item is a folder = video/frame folder
            if os.path.isdir(item_path):

                rows.append({
                    "path": item_path,
                    "label": label.replace("_", " "),
                    "type": task
                })

            # If Word contains images directly
            elif item.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):

                rows.append({
                    "path": label_path,
                    "label": label.replace("_", " "),
                    "type": task
                })


df = pd.DataFrame(rows)

csv_path = os.path.join(DATASET_PATH, "dataset.csv")

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("\nCSV saved successfully!")
print("Location:", csv_path)

print("\nTotal:", len(df))

print("\nWord and Sentence count:")
print(df["type"].value_counts())



CSV saved successfully!
Location: F:\ISL_CSLRT_Corpus\dataset.csv

Total: 1721

Word and Sentence count:
type
Word        1036
Sentence     685
Name: count, dtype: int64

Sample:
                              path   label  type
0   F:\ISL_CSLRT_Corpus\Word\A LOT   A LOT  Word
1   F:\ISL_CSLRT_Corpus\Word\A LOT   A LOT  Word
2   F:\ISL_CSLRT_Corpus\Word\A LOT   A LOT  Word
3   F:\ISL_CSLRT_Corpus\Word\A LOT   A LOT  Word
4   F:\ISL_CSLRT_Corpus\Word\ABUSE   ABUSE  Word
5   F:\ISL_CSLRT_Corpus\Word\ABUSE   ABUSE  Word
6   F:\ISL_CSLRT_Corpus\Word\ABUSE   ABUSE  Word
7   F:\ISL_CSLRT_Corpus\Word\ABUSE   ABUSE  Word
8   F:\ISL_CSLRT_Corpus\Word\ABUSE   ABUSE  Word
9  F:\ISL_CSLRT_Corpus\Word\AFRAID  AFRAID  Word
